# Feature Distribution Analysis - SiTunes dataset

This notebook analyzes the distributions, ranges, and outliers of extracted features to determine the appropriate scaling and encoding strategies for feature engineering.

In [ ]:
%matplotlib inline

import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
from typing import Any
from ydata_profiling import ProfileReport

In [ ]:
# Visualization settings
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (15, 10)

In [ ]:
# Add src to path to allow imports from src
ROOT_DIR = Path.cwd().parent.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

In [ ]:
from src.data.file_loader import load_csv_data, load_npy_data, save_csv_data

from src.utils.const import INTERIM_DATA_DIR, REPORTS_DIR, SiTunesInterimWristColsIdx
from src.data.const import (
    SITUNES_MERGED_MUSIC_METADATA_FILENAME as INTERIM_MUSIC_METADATA_FILENAME, 
    SITUNES_MERGED_INTER_ENV_DATA_FILENAME as INTERIM_INTER_ENV_DATA_FILENAME, 
    SITUNES_MERGED_WRIST_DATA_FILENAME as INTERIM_WRIST_DATA_FILENAME
)
from src.data.const import SITUNES_MUSIC_METADATA_CONFIG, SitunesConfigKeys

INTERIM_MUSIC_METADATA_PATH = INTERIM_DATA_DIR / INTERIM_MUSIC_METADATA_FILENAME
INTERIM_INTER_ENV_DATA_PATH = INTERIM_DATA_DIR / INTERIM_INTER_ENV_DATA_FILENAME
INTERIM_WRIST_DATA_PATH = INTERIM_DATA_DIR / INTERIM_WRIST_DATA_FILENAME

MUSIC_METADATA_PKEY = SITUNES_MUSIC_METADATA_CONFIG.get('music_info', {}).get(SitunesConfigKeys.PKEY_CANDIDATE_COLS, [None])[0]

YDATA_REPORT_NAME = "eda_feature_distribution_report.html"

FEATURES_STATS_CSV_NAME = "features_stats.csv"

In [ ]:
class NumericColsToAnalyze:
    """
    Constants for all numeric columns to be analyzed in EDA and scaling diagnostics.
    This includes biological, motion, context, and musical features.
    """
    # Biological features (Wrist HR)
    HR_MEAN: str = 'hr_mean'
    HR_STD: str = 'hr_std'
    HR_MIN: str = 'hr_min'
    HR_MAX: str = 'hr_max'
    HR_TREND_SLOPE: str = 'hr_trend_slope'
    HR_TREND_R_VALUE: str = 'hr_trend_r_value'
    HR_TREND_P_VALUE: str = 'hr_trend_p_value'
    HR_TREND_STD_ERR: str = 'hr_trend_std_err'
    HR_TREND_INTERCEPT: str = 'hr_trend_intercept'

    # Motion features - Intensity
    ACT_INTENSITY_MEAN: str = 'act_intensity_mean'
    ACT_INTENSITY_STD: str = 'act_intensity_std'
    ACT_INTENSITY_MIN: str = 'act_intensity_min'
    ACT_INTENSITY_MAX: str = 'act_intensity_max'

    # Motion features - Steps
    ACT_STEPS_MEAN: str = 'act_steps_mean'
    ACT_STEPS_STD: str = 'act_steps_std'
    ACT_STEPS_MIN: str = 'act_steps_min'
    ACT_STEPS_MAX: str = 'act_steps_max'
    ACT_STEPS_SUM: str = 'act_steps_sum'
    ACT_STEPS_ACTIVE_RATIO: str = 'act_steps_active_ratio'

    # Motion features - Types (Changes is numeric, Dominant is ID but useful for freq analysis)
    ACT_TYPE_DOMINANT: str = 'act_type_dominant'
    ACT_TYPE_CHANGES_COUNT: str = 'act_type_changes_count'

    # Context / Environment features
    WEATHER_PRESSURE: str = 'weather_pressure'
    WEATHER_TEMP: str = 'weather_temp'
    WEATHER_HUMIDITY: str = 'weather_humidity'
    GPS_SPEED: str = 'gps_speed'

    # Musical features - High level
    MUSIC_POPULARITY: str = 'popularity'
    MUSIC_LOUDNESS: str = 'loudness'
    MUSIC_DANCEABILITY: str = 'danceability'
    MUSIC_ENERGY: str = 'energy'
    MUSIC_SPEECHINESS: str = 'speechiness'
    MUSIC_ACOUSTICNESS: str = 'acousticness'
    MUSIC_INSTRUMENTALNESS: str = 'instrumentalness'
    MUSIC_VALENCE: str = 'valence'
    MUSIC_TEMPO: str = 'tempo'
    MUSIC_DURATION: str = 'duration_music'

    # Musical features - Low level (OpenSMILE)
    MUSIC_F0_MEAN: str = 'F0final_sma_amean'
    MUSIC_F0_STD: str = 'F0final_sma_stddev'
    MUSIC_RMS_STD: str = 'pcm_RMSenergy_sma_stddev'
    MUSIC_ZCR_MEAN: str = 'pcm_zcr_sma_amean'
    MUSIC_PSY_SHARPNESS_MEAN: str = 'pcm_fftMag_psySharpness_sma_amean'

    # Emotion features
    EMO_PRE_VALENCE: str = 'emo_pre_valence'
    EMO_PRE_AROUSAL: str = 'emo_pre_arousal'

    @classmethod
    def get_all_features(cls) -> list[str]:
        """Returns all defined numeric feature names as a list."""
        return [
            value for key, value in cls.__dict__.items() 
            if not key.startswith('__') and isinstance(value, str)
        ]

### 1. Data loading and preparation

The following block handles interim datasets loading and merging to a single `DataFrame`, ready for further analysis.

In [ ]:
def load_interim_datasets(
    music_metadata_path: Path = INTERIM_MUSIC_METADATA_PATH,
    inter_env_data_path: Path = INTERIM_INTER_ENV_DATA_PATH,
    wrist_data_path: Path = INTERIM_WRIST_DATA_PATH
) -> tuple[pd.DataFrame, pd.DataFrame, np.ndarray]:
    """Load interim datasets."""
    music_metadata_df = load_csv_data(music_metadata_path)
    inter_env_data_df = load_csv_data(inter_env_data_path)
    wrist_data_array = load_npy_data(wrist_data_path)
    
    return music_metadata_df, inter_env_data_df, wrist_data_array


def merge_all_datasets(
    inter_env_df: pd.DataFrame, 
    music_df: pd.DataFrame, 
    wrist_aggregated_df: pd.DataFrame,
    music_inter_merge_key: str = MUSIC_METADATA_PKEY
) -> pd.DataFrame:
    """
    Merge all feature sets into a single analysis DataFrame.
    
    Args:
        inter_env_df: Interaction and environment data.
        music_df: Music metadata.
        wrist_aggregated_df: Aggregated wrist features.

    Returns:
        pd.DataFrame: Merged dataset.
    """
    # Combine inter_env and wrist_aggregated_df by index (order-based alignment)
    combined_context_bio = pd.concat([inter_env_df, wrist_aggregated_df], axis=1)
    
    # Merge with music metadata on item_id
    full_df = combined_context_bio.merge(
        music_df, 
        on=music_inter_merge_key, 
        how="left",
        suffixes=('_inter_env', '_music')
    )
    
    return full_df

### 2. Wrist data aggregation

The code defined in `aggregate_wrist_data` function below is responsible for flattening interim wrist data nested in matrices. Multiple statistics are calculated for **heart rate**, **activity intensity**, **activity steps** and **activity type** data. 

In [ ]:
def aggregate_wrist_data(
    wrist_data_array: np.ndarray,
    hr_col_idx: int = SiTunesInterimWristColsIdx.HEART_RATE.value,
    act_intensity_col_idx: int = SiTunesInterimWristColsIdx.ACT_INTENSITY.value,
    act_steps_col_idx: int = SiTunesInterimWristColsIdx.ACT_STEPS.value,
    act_type_col_idx: int = SiTunesInterimWristColsIdx.ACT_TYPE.value
) -> pd.DataFrame:
    """Aggregate wrist data by calculating statistics for each wrist data feature."""
    aggregated_data: list[dict[str, Any]] = []
    time_axis = np.arange(wrist_data_array.shape[1])
    
    for i in range(wrist_data_array.shape[0]):
        sample = wrist_data_array[i]
        hr_signal = sample[:, hr_col_idx]
        act_intensity_signal = sample[:, act_intensity_col_idx]
        act_steps_signal = sample[:, act_steps_col_idx]
        act_type_signal = sample[:, act_type_col_idx]
        
        # Linear regression for heart rate signal trend
        slope, intercept, r_val, p_val, std_err = stats.linregress(time_axis, hr_signal)
        
        aggregated_data.append({
            NumericColsToAnalyze.HR_MEAN: np.mean(hr_signal),
            NumericColsToAnalyze.HR_STD: np.std(hr_signal),
            NumericColsToAnalyze.HR_MIN: np.min(hr_signal),
            NumericColsToAnalyze.HR_MAX: np.max(hr_signal),
            NumericColsToAnalyze.HR_TREND_SLOPE: slope,
            NumericColsToAnalyze.HR_TREND_R_VALUE: r_val,
            NumericColsToAnalyze.HR_TREND_P_VALUE: p_val,
            NumericColsToAnalyze.HR_TREND_STD_ERR: std_err,
            NumericColsToAnalyze.HR_TREND_INTERCEPT: intercept,
            NumericColsToAnalyze.ACT_INTENSITY_MEAN: np.mean(act_intensity_signal),
            NumericColsToAnalyze.ACT_INTENSITY_STD: np.std(act_intensity_signal),
            NumericColsToAnalyze.ACT_INTENSITY_MIN: np.min(act_intensity_signal),
            NumericColsToAnalyze.ACT_INTENSITY_MAX: np.max(act_intensity_signal),
            NumericColsToAnalyze.ACT_STEPS_MEAN: np.mean(act_steps_signal),
            NumericColsToAnalyze.ACT_STEPS_STD: np.std(act_steps_signal),
            NumericColsToAnalyze.ACT_STEPS_MIN: np.min(act_steps_signal),
            NumericColsToAnalyze.ACT_STEPS_MAX: np.max(act_steps_signal),
            NumericColsToAnalyze.ACT_STEPS_SUM: np.sum(act_steps_signal),
            NumericColsToAnalyze.ACT_STEPS_ACTIVE_RATIO: np.count_nonzero(act_steps_signal) / len(act_steps_signal),
            NumericColsToAnalyze.ACT_TYPE_DOMINANT: int(stats.mode(act_type_signal, keepdims=True).mode[0]),
            NumericColsToAnalyze.ACT_TYPE_CHANGES_COUNT: np.count_nonzero(np.diff(act_type_signal))
        })
        
    return pd.DataFrame(aggregated_data)

### 3. Visualization and statistical analysis functions

This code defines feature distribution analysis and scaling selection. The `plot_diagnostic_grid` function renders a 3-panel visualization (**histogram** with KDE, **boxplot**, and **Q-Q plot**) to evaluate data shape, outliers, and normality for a specific column. Complementing this, `get_scaling_stats_table` calculates key profile metrics for every feature: *skewness*, *kurtosis*, *min*, *max*, *std*, and a sample-capped *Shapiro-Wilk test*.

In [ ]:
def plot_diagnostic_grid(df: pd.DataFrame, column: str) -> None:
    """
    Plot a 3-panel diagnostic visualization for a feature:
    Histogram (Shape), Boxplot (Outliers), and QQ-Plot (Normality).
    """
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # 1. Distribution
    sns.histplot(df[column], kde=True, ax=axes[0], color='teal')
    axes[0].set_title(f'Distribution: {column}')
    
    # 2. Outliers
    sns.boxplot(x=df[column], ax=axes[1], color='salmon')
    axes[1].set_title(f'Outliers: {column}')
    
    # 3. QQ-Plot
    stats.probplot(df[column], dist="norm", plot=axes[2])
    axes[2].set_title(f'QQ-Plot (Normality): {column}')
    
    plt.tight_layout()
    plt.show()

def get_scaling_stats_table(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    """
    Calculate key metrics to decide on scaling.
    
    High Skewness (> 1 or < -1) or High Kurtosis (> 3) suggests StandardScaler.
    Low skewness and bounded ranges might favor MinMaxScaler.
    """
    stats_list = []
    for col in columns:
        col_data = df[col].dropna()
        stats_list.append({
            "feature": col,
            "skewness": col_data.skew(),
            "kurtosis": col_data.kurtosis(),
            "min": col_data.min(),
            "max": col_data.max(),
            "std": col_data.std(),
            "is_normal_shapiro": stats.shapiro(col_data)[1] > 0.05 if len(col_data) < 5000 else "N/A"
        })
    
    return pd.DataFrame(stats_list).set_index("feature")

### 4. Data report generation using `ydata-profiling`

In [ ]:
def generate_profile_report(
    df: pd.DataFrame, 
    report_title: str = YDATA_REPORT_NAME,
    report_filepath: Path = REPORTS_DIR / YDATA_REPORT_NAME
) -> None:
    """Generate a comprehensive profiling report using ydata-profiling."""
    profile = ProfileReport(df, title=report_title, explorative=True)
    profile.to_file(report_filepath)
    print(f"Profiling report saved to: {report_filepath}")

### 5. Analysis execution

In [ ]:
# Load and aggregate interim data
music_metadata_df, inter_env_data_df, wrist_data_array = load_interim_datasets()
aggregated_wrist_df = aggregate_wrist_data(wrist_data_array)
merged_df = merge_all_datasets(inter_env_data_df, music_metadata_df, aggregated_wrist_df)

# Select only numeric columns for analysis
merged_df_to_analyze = merged_df[NumericColsToAnalyze.get_all_features()]
print(merged_df_to_analyze.head())

# Generate profiling report
generate_profile_report(merged_df_to_analyze)

# Get feature statistics
scaling_stats_table = get_scaling_stats_table(merged_df_to_analyze, NumericColsToAnalyze.get_all_features())
print(scaling_stats_table.head())
save_csv_data(scaling_stats_table, REPORTS_DIR / FEATURES_STATS_CSV_NAME, index=True)

# Display diagnostic plots for every feature
for feature in NumericColsToAnalyze.get_all_features():
    plot_diagnostic_grid(merged_df_to_analyze, feature)